In [3]:
import os
import glob
import shutil
import time
import cv2
import numpy as np
from deepface import DeepFace
import tensorflow as tf

# Configure TensorFlow to use GPU
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        # Currently, memory growth needs to be the same across GPUs
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        
        # Restrict TensorFlow to only use the first GPU
        tf.config.experimental.set_visible_devices(gpus[0], 'GPU')
        
        print(f"GPU support enabled. Using: {gpus[0].name}")
    except RuntimeError as e:
        print(f"GPU configuration error: {e}")
else:
    print("No GPU found. Using CPU instead.")

def setup_face_database():
    # Directory containing known face images
    known_faces_dir = "faces/test"
    known_face_paths = glob.glob(os.path.join(known_faces_dir, "*.jpg"))
    
    if len(known_face_paths) == 0:
        print(f"No images found in {known_faces_dir}")
        return None
    
    print(f"Loading {len(known_face_paths)} known face images")
    
    # Create a database for DeepFace
    temp_db_path = "temp_face_db"
    os.makedirs(temp_db_path, exist_ok=True)
    
    # Process known faces and organize them for recognition
    for face_path in known_face_paths:
        person_name = os.path.basename(face_path).split('.')[0]
        person_dir = os.path.join(temp_db_path, person_name)
        os.makedirs(person_dir, exist_ok=True)
        
        # Copy the image to the person's directory
        dest_path = os.path.join(person_dir, os.path.basename(face_path))
        if not os.path.exists(dest_path):
            shutil.copy(face_path, dest_path)
    
    print("Face database prepared successfully")
    return temp_db_path

def cuda_optimized_face_recognition():
    # Setup the face database
    db_path = setup_face_database()
    if not db_path:
        print("Failed to setup face database. Exiting.")
        return
    
    # Initialize webcam
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Error: Could not open webcam")
        return
    
    # Configuration for optimal CUDA performance
    model_name = "ArcFace"  # GPU-optimized model
    distance_metric = "cosine"
    detector_backend = "retinaface"  # GPU-optimized detector
    
    # Performance metrics
    frame_count = 0
    start_time = time.time()
    fps_update_interval = 10  # Update FPS every 10 frames
    process_this_frame = True
    last_process_time = time.time()
    
    # GPU-optimized batch size for processing
    batch_size = 1  # Adjust based on your GPU memory
    
    # Initialize representations to avoid recomputing for database
    db_representations = []
    db_identities = []
    
    print("Pre-computing representations for known faces...")
    for person_folder in os.listdir(db_path):
        person_path = os.path.join(db_path, person_folder)
        if os.path.isdir(person_path):
            for img_file in os.listdir(person_path):
                if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                    img_path = os.path.join(person_path, img_file)
                    try:
                        embedding = DeepFace.represent(
                            img_path=img_path,
                            model_name=model_name,
                            detector_backend=detector_backend,
                            enforce_detection=False,
                            align=True
                        )
                        db_representations.append(embedding)
                        db_identities.append(person_folder)
                    except Exception as e:
                        print(f"Error processing {img_path}: {e}")
    
    print(f"Pre-computed {len(db_representations)} face representations")
    
    try:
        print("Starting GPU-accelerated face recognition. Press Q to quit.")
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            current_time = time.time()
            elapsed_time = current_time - start_time
            
            # Process at reasonable intervals for better GPU utilization
            if process_this_frame and (current_time - last_process_time > 0.1):  # 10 FPS max for processing
                last_process_time = current_time
                
                try:
                    # Extract faces using GPU acceleration
                    faces = DeepFace.extract_faces(
                        img_path=frame,
                        detector_backend=detector_backend,
                        enforce_detection=False,
                        align=True
                    )
                    
                    for face in faces:
                        facial_area = face["facial_area"]
                        x, y, w, h = facial_area["x"], facial_area["y"], facial_area["w"], facial_area["h"]
                        
                        # Draw rectangle around the face
                        cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
                        
                        # Extract the face region
                        face_img = frame[y:y+h, x:x+w]
                        
                        try:
                            # Get face representation using GPU
                            target_representation = DeepFace.represent(
                                img_path=face_img,
                                model_name=model_name,
                                detector_backend=detector_backend,
                                enforce_detection=False,
                                align=True
                            )
                            
                            # Find the most similar face in our database
                            min_distance = float('inf')
                            best_match = None
                            
                            for i, db_rep in enumerate(db_representations):
                                # Calculate similarity
                                if distance_metric == "cosine":
                                    distance = DeepFace.dst.findCosineDistance(db_rep, target_representation)
                                elif distance_metric == "euclidean":
                                    distance = DeepFace.dst.findEuclideanDistance(db_rep, target_representation)
                                elif distance_metric == "euclidean_l2":
                                    distance = DeepFace.dst.findEuclideanDistance(
                                        DeepFace.dst.l2_normalize(db_rep),
                                        DeepFace.dst.l2_normalize(target_representation)
                                    )
                                
                                if distance < min_distance:
                                    min_distance = distance
                                    best_match = db_identities[i]
                            
                            # Threshold for recognition
                            threshold = 0.5  # Adjust based on your needs and chosen model
                            
                            if min_distance < threshold and best_match:
                                label = f"{best_match} ({min_distance:.2f})"
                                color = (0, 255, 0)  # Green for recognized
                            else:
                                label = f"Unknown ({min_distance:.2f})"
                                color = (0, 0, 255)  # Red for unknown
                            
                            # Display name and confidence
                            cv2.putText(
                                frame, label, (x, y-10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2
                            )
                        except Exception as e:
                            print(f"Error processing face: {e}")
                
                except Exception as e:
                    print(f"Frame processing error: {e}")
            
            # Update FPS counter
            frame_count += 1
            if frame_count % fps_update_interval == 0:
                fps = frame_count / elapsed_time
                frame_count = 0
                start_time = time.time()
                print(f"FPS: {fps:.2f}")
            
            # Toggle processing flag (process every other frame)
            process_this_frame = not process_this_frame
            
            # Add GPU indicator
            if gpus:
                cv2.putText(
                    frame, "GPU Accelerated", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2
                )
            
            # Display the result
            cv2.imshow('CUDA Face Recognition', frame)
            
            # Exit on 'q' key press
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
    
    finally:
        # Clean up
        cap.release()
        cv2.destroyAllWindows()
        if os.path.exists(db_path):
            shutil.rmtree(db_path)

if __name__ == "__main__":
    cuda_optimized_face_recognition()


No GPU found. Using CPU instead.
Loading 4 known face images
Face database prepared successfully
Pre-computing representations for known faces...


KeyboardInterrupt: 

In [5]:
import torch
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda:0


In [ ]:
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Then import TensorFlow
import tensorflow as tf
print(tf.config.list_physical_devices("GPU"))


In [ ]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPU Available:", len(tf.config.list_physical_devices("GPU")) > 0)
print("GPU Devices:", tf.config.list_physical_devices("GPU"))

# Also check PyTorch for comparison
import torch
print("PyTorch version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
